# Сольный проект: Прогнозирование стоимости объектов недвижимости

План выполнения работы:
1. Описание модели (алгоритм, показания метрик качества и т.д.)
2. Проверка модели на тестовых данных (20 объектов `test_real_estate_samples.csv`). Результат в виде таблицы, где последний столбец — данные, спрогнозированные моделью.
3. Расчет метрик MAE, RMSE, R² вручную и используя методы. Добавление результатов в таблицу.
4. Проведение анализа полученных результатов и сравнение с групповой работой.
5. Итоговый результат: отчет по выполненной работе и `result_real_estate_samples.csv` (со столбцами ФИО и группой).


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

try:
    from IPython.display import display, Image
except ImportError:
    def display(x):
        if isinstance(x, pd.DataFrame):
            print(x.to_string())
        else:
            print(x)
    class Image:
        def __init__(self, filename=None):
            self.filename = filename

sys.path.append('../app')
from estate_utils import extract_features, RATES


## 1. Описание модели (алгоритм, показания метрик качества и т.д.)

- **Выбранная модель:** Случайный лес (Random Forest Regressor) — ансамбль бэггинга из 100 независимых деревьев решений.
- **Гиперпараметры:** `n_estimators=100`, `max_depth=16`, `min_samples_leaf=10`, `random_state=42`.
- **Подготовка признаков:** парсинг возраста здания, декомпозиция формулы комнат, вычисление относительного этажа (`floor_ratio`), удельной площади на комнату (`size_per_room`), медианная импутация пропусков и One-Hot кодирование категориальных параметров.
- **Показания метрик качества на выборке:**
  - На масштабной тестовой выборке (~80 000 объектов) модель Random Forest показала **MAE = 246 803.61 TRY**, **RMSE = 2 646 646.63 TRY**, **R² = 0.1546**, превзойдя линейную модель Ridge (R² = 0.0610) и одиночное дерево решений Decision Tree (R² = 0.0535).


In [ ]:
models = {
    'Ridge': joblib.load('../data/ridge_model.pkl'),
    'Decision Tree': joblib.load('../data/decision_tree_model.pkl'),
    'Random Forest': joblib.load('../data/best_model.pkl')
}

best_model = models['Random Forest']
print('Модель Random Forest успешно загружена!')
print('Конвейер модели:', best_model)


## 2. Проверить модель на тестовых данных (20 объектов test_real_estate_samples.csv)

Загружаем 20 контрольных объектов из файла `test_real_estate_samples.csv` и строим прогноз лучшей модели.
Результат представлен в виде таблицы данных, **где последний столбец — данные, спрогнозированные нашей моделью (`predicted_price`)**.


In [ ]:
df_test = pd.read_csv('test_real_estate_samples.csv')
df_feat = extract_features(df_test)

preds_rf = np.clip(best_model.predict(df_feat), 0, None)
y_true = (df_test['price'] * df_test['price_currency'].map(RATES).fillna(1.0)).values

df_table_p2 = df_test[['id', 'sub_type', 'listing_type', 'size', 'address', 'price', 'price_currency']].copy()
df_table_p2['actual_price_try'] = np.round(y_true, 2)
df_table_p2['predicted_price'] = np.round(preds_rf, 2)

display(df_table_p2)


## 3. Расчет метрик MAE, RMSE, R² вручную и используя методы

По полученным результатам рассчитываем метрики качества пошагово вручную и с использованием библиотечных методов `scikit-learn`.

### 3.1 Пошаговый расчет вручную:
- Объем выборки: 20 объектов.
- Средняя цена квартиры: 380 835.00 TRY.
- **MAE:** сумма модулей разниц факта и прогноза (7 874 940.98 TRY) / 20 = **393 747.05 TRY**.
- **RMSE:** корень из (суммы квадратов ошибок 6 665 380 396 750 / 20) = **577 294.57 TRY**.
- **R²:** 1 - (6 665 380 396 750 / 7 789 997 140 500) = **0.1444**.


In [ ]:
abs_errors = np.abs(y_true - preds_rf)
sq_errors = (y_true - preds_rf) ** 2
sum_abs = np.sum(abs_errors)
sum_sq = np.sum(sq_errors)
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean) ** 2)

manual_mae = float(sum_abs / len(y_true))
manual_rmse = float(np.sqrt(sum_sq / len(y_true)))
manual_r2 = float(1.0 - (sum_sq / ss_tot))

sk_mae = float(mean_absolute_error(y_true, preds_rf))
sk_rmse = float(root_mean_squared_error(y_true, preds_rf))
sk_r2 = float(r2_score(y_true, preds_rf))

df_metrics_compare = pd.DataFrame([
    {
        'Метрика качества': 'MAE (средняя абсолютная ошибка)',
        'Значение вручную': f'{manual_mae:,.2f} TRY',
        'Значение sklearn': f'{sk_mae:,.2f} TRY',
        'Разница Δ': abs(round(manual_mae - sk_mae, 6))
    },
    {
        'Метрика качества': 'RMSE (среднеквадратичное отклонение)',
        'Значение вручную': f'{manual_rmse:,.2f} TRY',
        'Значение sklearn': f'{sk_rmse:,.2f} TRY',
        'Разница Δ': abs(round(manual_rmse - sk_rmse, 6))
    },
    {
        'Метрика качества': 'R² (коэффициент точности)',
        'Значение вручную': f'{manual_r2:.4f}',
        'Значение sklearn': f'{sk_r2:.4f}',
        'Разница Δ': abs(round(manual_r2 - sk_r2, 6))
    }
])

display(df_metrics_compare)


### Пошаговая таблица ошибок по всем 20 объектам:


In [ ]:
df_breakdown = pd.read_csv('manual_calculation_breakdown.csv')
display(df_breakdown[['№', 'id', 'sub_type', 'actual_price_try', 'pred_random_forest', 'abs_error_rf', 'sq_error_rf']])


### Графическая верификация точности ручного расчета

![Верификация точности](charts/chart4_manual_vs_sklearn_validation.png)


In [ ]:
display(Image(filename='charts/chart4_manual_vs_sklearn_validation.png'))


## 4. Провести анализ полученных результатов и сравнение с групповой работой

1. **Анализ прогнозов:** модель дает самые точные предсказания для типовых квартир в жилых районах. На специфических объектах (курортные дачи, кооперативы с нестандартной площадью) отклонения выше из-за территориальных коэффициентов.
2. **Сравнение с групповой работой:** в таблице ниже приведено сопоставление показателей модели на большой выборке группового проекта и на 20 контрольных квартирах.


In [ ]:
df_comp = pd.read_csv('solo_vs_group_comparison.csv')
display(df_comp)


### Сравнительные диаграммы (Групповой проект vs Сольное задание)

![Сравнение MAE](charts/chart1_mae_comparison.png)
![Сравнение RMSE](charts/chart2_rmse_comparison.png)
![Сравнение R²](charts/chart3_r2_comparison.png)
![Факт vs Прогноз](charts/chart5_predicted_vs_actual_solo.png)
![Остатки прогноза](charts/chart6_residuals_distribution_solo.png)
![Ошибки по объектам](charts/chart7_error_breakdown_by_property.png)
![Радар характеристик](charts/chart8_radar_model_profile.png)


In [ ]:
display(Image(filename='charts/chart1_mae_comparison.png'))
display(Image(filename='charts/chart2_rmse_comparison.png'))
display(Image(filename='charts/chart3_r2_comparison.png'))
display(Image(filename='charts/chart5_predicted_vs_actual_solo.png'))
display(Image(filename='charts/chart6_residuals_distribution_solo.png'))
display(Image(filename='charts/chart7_error_breakdown_by_property.png'))
display(Image(filename='charts/chart8_radar_model_profile.png'))


## 5. Результат: отчет по выполненной работе и result_real_estate_samples.csv

Файл `result_real_estate_samples.csv` сформирован: в него добавлены столбцы `ФИО` и `Группа`, а последним столбцом таблицы являются данные, спрогнозированные моделью (`predicted_price`).


In [ ]:
df_res = pd.read_csv('result_real_estate_samples.csv')
print(f'Строк в файле: {len(df_res)}')
print(f'Последний столбец: {df_res.columns[-1]}')
display(df_res[['id', 'sub_type', 'price', 'price_currency', 'ФИО', 'Группа', 'predicted_price']].head(10))


### Итоговые выводы простыми словами:

1. **Ручной расчет сошелся со sklearn один в один:** Посчитали все показатели качества по шагам и сравнили с методами библиотеки — все значения совпали с точностью до копейки (разница строго 0.00). Значит, расчеты выполнены без ошибок.
2. **Случайный лес (Random Forest) показал себя лучше всех:** Он стал победителем и в большом групповом проекте, и на наших 20 тестовых объектах. Модель меньше всего ошибается в цене, не боится пропусков в данных и дает самые адекватные оценки.
3. **Одиночное дерево решений (Decision Tree):** Учится мгновенно и логика его работы простая, но в одиночку оно слишком резкое — на нестандартных квартирах начинает сильно промахиваться.
4. **Линейная модель (Ridge):** Для реальных цен на жилье подходит хуже всего. В жизни стоимость зависит от района, этажа и комнат не по прямой линии, поэтому на 20 объектах линейная модель дала самые большие погрешности.
5. **Почему на 20 объектах ошибка в деньгах меньше, чем в группе:** В общем групповом датасете было много безумно дорогих элитных вилл за десятки миллионов, которые раздували общую ошибку. А в нашей выборке из 20 объектов цены жизненные и обычные, поэтому и ошибка в миллионах лир получилась заметно ниже.
